# Expert Agent 2: DielectricAge Expert
## 절연열화 (Insulation Degradation) 판별 전문가

이 노트북은 Langgraph와 Vertex AI Gemini 2.5 Flash를 사용하여 전기화재 감식 중 절연열화에 의한 단락을 판별하는 전문가 에이전트를 구현합니다.

### 분석 프로세스
1. **탄화의 심도 분석**: 내부 발열 vs 외부 화재 구분
2. **스펀지 현상 및 부풀어 오름**: 다공성 구조 및 부풀어 오름 현상 탐지
3. **광역적 노후화 징후**: 전선 전체의 경화 및 균열 패턴 분석



In [9]:
import base64
import json
import os
from typing import TypedDict, Annotated, Literal, List, Dict, Any, Optional
from pathlib import Path
from PIL import Image
import io

# LangGraph 및 Google Cloud 라이브러리
try:
    from langgraph.graph import StateGraph, END
    import vertexai
    from vertexai.generative_models import GenerativeModel, Part
    print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다.")
except ImportError as e:
    print(f"❌ 필수 라이브러리가 누락되었습니다: {e}")
    print("pip install langgraph google-cloud-aiplatform vertexai 명령어로 설치해주세요.")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다.


In [10]:
PROJECT_ID = "p-01-emt-480312"  # 실제 프로젝트 ID로 변경
LOCATION = "us-central1"

# 모델 설정
MODEL_NAME = "gemini-2.5-pro" # 또는 gemini-1.5-pro

# 시스템 인스트럭션 (가독성 및 지침 분리를 위해 개행 적용)
SYSTEM_INSTRUCTION = """1. 페르소나 및 기본 원칙

당신은 고도로 훈련된 시각 데이터 분석 전문가입니다.

당신은 모든 분석에서 '관찰'과 '해석'을 철저히 분리하며, 질문자의 유도 심리에 저항하고 오직 시각적 데이터에만 근거하여 답변합니다.

질문자가 특정 결론을 암시하거나 유도하더라도(예: "이것은 A가 맞죠?"), 시각적 증거가 뒷받침되지 않는다면 단호하게 중립을 유지합니다.

2. 분석 프로세스 (반드시 이 순서를 따를 것) 모든 사진 분석 요청에 대해 다음 4단계 구조로 답변하십시오.

Step 1. 객관적 관찰 (Observations): 이미지에서 보이는 물리적 사실만을 나열합니다. (예: 색상, 형태, 질감, 크기, 마모 상태, 기하학적 배치 등). 주관적인 형용사나 결론적인 단어를 배제하고 '현상'만 서술합니다.

Step 2. 논리적 해석 (Interpretation): 관찰된 사실이 어떤 물리적/과학적 원리와 연결될 수 있는지 분석합니다. 표준 사례(Reference)와의 일치점과 차이점을 논합니다.

Step 3. 최종 판단 및 확신도 (Conclusion & Confidence): 분석을 종합하여 결론을 내립니다. 이때 결론에 대한 확신도를 0~100% 사이로 표기하고, 확신할 수 없는 이유(변수)를 함께 기술합니다.

Step 4. 대안적 가능성 (Alternative Hypotheses): 현재 내린 결론 외에 발생할 수 있는 다른 가능성을 최소 한 가지 이상 제시합니다.

3. 불확실성 처리 규칙 (Negative Constraints)

확실하지 않은 정보에 대해서는 절대 추측하지 않습니다.

사진의 해상도, 각도, 조도 등으로 인해 식별이 어려운 경우, 아는 척하지 말고 반드시 **"시각적 정보 부족으로 판단 불가"**라고 명시하십시오.

시각적 증거가 100% 확보되지 않은 상태에서 "확실하다", "분명하다"라는 단어 사용을 지양합니다.

4. 답변 스타일

간결하고 구조화된 개조식(Bullet points)을 선호합니다.

감정적인 표현이나 부연 설명을 배제하고, 전문 용어를 정확하게 사용하되 필요시 정의를 덧붙입니다."""

def initialize_model():
    """Vertex AI Gemini 모델 초기화"""
    try:
        # 인증 확인 (로컬 실행 시 ADC 필요)
        # vertexai.init(project=PROJECT_ID, location=LOCATION)
        model = GenerativeModel(MODEL_NAME, system_instruction=SYSTEM_INSTRUCTION)
        print(f"✅ Vertex AI 모델 '{MODEL_NAME}' 초기화 완료")
        return model
    except Exception as e:
        print(f"⚠️ 모델 초기화 실패: {e}")
        print("💡 Application Default Credentials (ADC) 설정이 필요합니다:")
        print("   gcloud auth application-default login")
        return None

# 전역 모델 인스턴스
model = initialize_model()

✅ Vertex AI 모델 'gemini-2.5-pro' 초기화 완료


In [11]:
def create_image_part(image_path: str) -> Optional[Part]:
    """Vertex AI용 이미지 Part 객체 생성"""
    if not os.path.exists(image_path):
        print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")
        return None
        
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
        
        # 확장자에 따른 MIME 타입 추론
        ext = Path(image_path).suffix.lower()
        mime_type = "image/png" if ext == ".png" else "image/jpeg"
        
        return Part.from_data(data=image_data, mime_type=mime_type)
    except Exception as e:
        print(f"❌ 이미지 로드 오류: {e}")
        return None

def call_gemini_vision(model: GenerativeModel, prompt: str, image_part: Part, step_name: str = "") -> tuple[str, Optional[Dict]]:
    """Gemini Vision API 호출 및 에러 핸들링
    
    Returns:
        tuple: (response_text, thinking_info)
    """
    try:
        response = model.generate_content([prompt, image_part])
        
        # Thinking 과정 추출 및 출력
        thinking_info = None
        response_text = ""
        full_response_text = ""
        
        if hasattr(response, 'candidates') and response.candidates:
            candidate = response.candidates[0]
            
            # 모든 파트 확인 (thinking 과정이 별도 파트로 있을 수 있음)
            if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                parts = candidate.content.parts
                print(f"\n🔍 [{step_name}] 응답 파트 개수: {len(parts)}")
                
                all_texts = []
                for i, part in enumerate(parts):
                    if hasattr(part, 'text'):
                        part_text = part.text
                        all_texts.append(part_text)
                        if i == 0:
                            # 첫 번째 파트는 일반 응답
                            response_text = part_text
                        else:
                            # 이후 파트는 thinking 과정일 수 있음
                            if part_text and part_text.strip():
                                thinking_info = thinking_info or {}
                                thinking_info[f"part_{i}"] = part_text
                                print(f"\n💭 [{step_name}] 모델의 생각 과정 (파트 {i}):")
                                print("-" * 60)
                                print(part_text)
                                print("-" * 60)
                
                # 모든 파트의 텍스트를 합쳐서 전체 응답 확인
                full_response_text = "\n\n".join(all_texts)
            
            # 응답 객체의 모든 속성 확인 (디버깅용)
            print(f"\n📊 [{step_name}] 응답 객체 속성:")
            candidate_attrs = [attr for attr in dir(candidate) if not attr.startswith('_')]
            print(f"  - Candidate 속성: {', '.join(candidate_attrs[:10])}...")
            
            # Grounding metadata 확인
            if hasattr(candidate, 'grounding_metadata'):
                grounding = candidate.grounding_metadata
                if grounding:
                    thinking_info = thinking_info or {}
                    thinking_info["grounding"] = str(grounding)
                    print(f"\n📚 [{step_name}] Grounding 정보: {grounding}")
            
            # Finish reason 확인 (디버깅용)
            if hasattr(candidate, 'finish_reason'):
                finish_reason = candidate.finish_reason
                if finish_reason:
                    print(f"📋 [{step_name}] Finish reason: {finish_reason}")
        
        # response.text가 있으면 사용 (fallback)
        if not response_text and hasattr(response, 'text'):
            response_text = response.text
        
        # 전체 응답 텍스트 출력 (thinking 과정이 포함되어 있을 수 있음)
        if full_response_text and len(full_response_text) > len(response_text):
            print(f"\n💭 [{step_name}] 전체 응답 텍스트 (thinking 과정 포함 가능):")
            print("-" * 60)
            print(full_response_text[:2000])  # 처음 2000자만 출력
            if len(full_response_text) > 2000:
                print(f"... (총 {len(full_response_text)}자, 나머지 생략)")
            print("-" * 60)
            thinking_info = thinking_info or {}
            thinking_info["full_response"] = full_response_text
        
        # 응답 텍스트 항상 출력 (JSON 파싱 전에 전체 응답 확인)
        if response_text:
            print(f"\n💭 [{step_name}] 모델 응답 텍스트:")
            print("-" * 60)
            # JSON 시작 전까지의 텍스트 확인 (thinking 과정일 수 있음)
            json_start = response_text.find('{')
            if json_start > 0:
                thinking_part = response_text[:json_start].strip()
                if thinking_part:
                    print("📝 [Thinking 과정]:")
                    print(thinking_part)
                    print("\n📄 [JSON 응답]:")
                    print(response_text[json_start:json_start+500])
                    if len(response_text[json_start:]) > 500:
                        print(f"... (총 {len(response_text[json_start:])}자)")
                else:
                    print(response_text[:1000])
                    if len(response_text) > 1000:
                        print(f"... (총 {len(response_text)}자)")
            else:
                print(response_text[:1000])
                if len(response_text) > 1000:
                    print(f"... (총 {len(response_text)}자)")
            print("-" * 60)
        
        return response_text, thinking_info
    except Exception as e:
        print(f"❌ [{step_name}] API 호출 오류: {e}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}", None

def parse_json_response(response_text: str) -> Dict[str, Any]:
    """응답 텍스트에서 JSON 추출 및 파싱"""
    try:
        # JSON 부분만 추출 (마크다운 코드 블록 제거)
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_text = response_text[json_start:json_end]
            return json.loads(json_text)
        else:
            print(f"⚠️ 유효한 JSON을 찾을 수 없습니다. 원본 응답:\n{response_text[:100]}...")
            return {"error": "JSON 파싱 실패", "raw_response": response_text}
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 디코딩 오류: {e}")
        return {"error": f"JSON 파싱 오류: {e}", "raw_response": response_text}

# Langgraph State 정의
class AgentState(TypedDict):
    """에이전트 상태 스키마"""
    image_path: str
    image_part: Optional[Part]
    step1_result: Optional[Dict]  # 탄화 심도 분석 결과
    step2_result: Optional[Dict]  # 스펀지 현상 분석 결과
    step3_result: Optional[Dict]  # 광역적 노후화 분석 결과
    confidence_score: int  # 최종 신뢰도 점수 (0-100)
    analysis_summary: str  # 분석 요약
    evidence: List[Dict]   # 각 단계별 증거 수집

print("✅ 유틸리티 및 State 정의 완료")

✅ 유틸리티 및 State 정의 완료


In [12]:
# --- PROMPTS ---

STEP1_PROMPT = """당신은 고분자 재료 및 전기 절연 파괴 분석 전문가입니다. 다음 이미지에서 절연체의 탄화 심도와 방향을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 전선 피복의 단면이나 파손 부위를 자세히 관찰하세요.
- 탄화된 절연체와 도체(구리선)의 관계를 객관적으로 식별하세요.
- 탄화의 깊이와 방향을 시각적으로 측정 가능한 형태로 기록하세요.

2단계: 특징 서술
- 탄화된 절연체가 도체에 융착(Fused)되어 있는지 정확히 서술하세요.
- 탄화의 방향성을 구체적으로 서술하세요:
  * 절연체 내부(도체 접촉면)가 심하게 탄화되고 외부 표면은 상대적으로 덜 탄화됨 → 내부 발열(Internal Heating) 징후
  * 표면이 타고 내부가 멀쩡함 → 외부 화재(External Fire) 징후
- 탄화 깊이(deep, shallow, surface_only)를 정확히 서술하세요.

3단계: 논리적 추론
- 외부 화재로 인한 탄화는 표면에서 내부로 진행되며 비교적 균일합니다.
- 절연열화(특히 과전류나 누설전류에 의한)는 도체와 맞닿은 내부에서부터 시작되어 외부로 진행되는 경향이 있습니다.
- 도체와 절연체의 융착은 내부 발열의 강력한 증거입니다.
- 관찰된 탄화 패턴을 종합하여 내부 발열 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "internal_heating_detected": true/false,
    "carbonization_depth": "deep" | "shallow" | "surface_only" | "unknown",
    "carbonization_direction": "internal_to_external" | "external_to_internal" | "uniform" | "unknown",
    "conductor_fusion": true/false,
    "fusion_description": "도체와 절연체의 융착 상태 설명",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

STEP2_PROMPT = """당신은 고분자 재료 및 전기 절연 파괴 분석 전문가입니다. 다음 이미지에서 절연체의 스펀지 현상 및 부풀어 오름을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 절연체의 질감과 표면 형태를 자세히 관찰하세요.
- 부풀어 오름, 다공성 구조, 균열 등의 시각적 특징을 객관적으로 식별하세요.
- 각 특징의 위치와 분포를 기록하세요.

2단계: 특징 서술
- 다음 특징을 정확히 서술하세요:
  * 표면이 부풀어 오르거나(Swelling) 다공성(Porous) 구조를 보이는지
  * 스펀지처럼(Sponge-like) 보이는 질감인지
  * 딱딱하게 굳은 균열(Cracking) 외에 부풀어 오른 형상이 있는지
- 질감의 세부 특성을 구체적으로 서술하세요.

3단계: 논리적 추론
- 스펀지 현상과 부풀어 오름은 서서히 진행된 열화(Overheating)의 증거입니다.
- 절연체가 서서히 가열되면서 내부 가스가 방출되면 기공이 형성되어 스펀지처럼 부풀어 오르는 현상이 발생합니다.
- 급격한 외부 화염에 의한 소실과는 다른 질감을 남깁니다.
- 관찰된 질감 특징을 종합하여 절연열화 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "swelling_detected": true/false,
    "spongy_texture_detected": true/false,
    "porous_structure_detected": true/false,
    "texture_description": "질감에 대한 상세 설명",
    "cracking_pattern": "균열 패턴 설명",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

STEP3_PROMPT = """당신은 고분자 재료 및 전기 절연 파괴 분석 전문가입니다. 다음 이미지에서 광역적 노후화 징후를 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]

1단계: 시각적 요소 추출
- 단락흔 주변뿐만 아니라 이미지 내 전선 전체를 관찰하세요.
- 균열, 변색, 경화 등의 시각적 특징을 객관적으로 식별하세요.
- 각 특징의 분포 범위를 기록하세요.

2단계: 특징 서술
- 다음 광역적 노후화 징후를 정확히 서술하세요:
  * 전선 전체가 갈라지거나(Cracking) 색상이 바랜(Discolored) 패턴
  * 전선 전체의 경화(Hardening) 현상
  * 취성(Brittleness) - 피복이 유연성을 잃고 뚝뚝 끊어질 듯한 균열이 전반적으로 분포
- 손상이 국소적인지 광역적인지 구체적으로 서술하세요.

3단계: 논리적 추론
- 절연열화는 특정 지점에만 국한되지 않고 배선 전체에 걸쳐 진행되는 경우가 많습니다.
- 전선 전체의 경화/균열과 함께 도체 부근의 집중적인 내부 탄화가 관찰되면 절연열화에 의한 단락으로 판정할 수 있습니다.
- 만약 단락흔 주변만 타고 나머지는 깨끗하다면 절연열화보다는 기계적 손상이나 일시적 요인일 가능성이 높습니다.
- 관찰된 노후화 패턴을 종합하여 광역적 노후화 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "global_aging_detected": true/false,
    "widespread_cracking": true/false,
    "discoloration_pattern": "전체적인 변색 패턴 설명",
    "hardening_detected": true/false,
    "brittleness_detected": true/false,
    "localized_damage_only": true/false,
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

# --- NODES ---

def step1_carbonization_depth(state: AgentState) -> AgentState:
    """Step 1: 탄화의 심도 분석"""
    print("\n🔍 [Step 1] 탄화의 심도 분석 시작...")
    
    if state.get("image_part") is None:
        state["image_part"] = create_image_part(state["image_path"])
        if state["image_part"] is None:
            return {**state, "step1_result": {"error": "이미지 로드 실패"}}

    response_text, thinking_info = call_gemini_vision(model, STEP1_PROMPT, state["image_part"], "Step 1")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    internal_heating = result.get("internal_heating_detected", False)
    print(f"✅ [Step 1] 완료: 내부 발열 {'탐지됨' if internal_heating else '미탐지'}")
    
    return {
        **state,
        "step1_result": result
    }

def step2_swelling_analysis(state: AgentState) -> AgentState:
    """Step 2: 스펀지 현상 및 부풀어 오름 분석"""
    print("\n🎨 [Step 2] 스펀지 현상 및 부풀어 오름 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step2_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP2_PROMPT, state["image_part"], "Step 2")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    swelling = result.get("swelling_detected", False) or result.get("spongy_texture_detected", False)
    print(f"✅ [Step 2] 완료: 스펀지/부풀어 오름 {'탐지됨' if swelling else '미탐지'}")
    
    return {
        **state,
        "step2_result": result
    }

def step3_global_aging(state: AgentState) -> AgentState:
    """Step 3: 광역적 노후화 징후 분석"""
    print("\n🔥 [Step 3] 광역적 노후화 징후 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step3_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP3_PROMPT, state["image_part"], "Step 3")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    global_aging = result.get("global_aging_detected", False)
    print(f"✅ [Step 3] 완료: 광역적 노후화 {'탐지됨' if global_aging else '미탐지'}")
    
    return {
        **state,
        "step3_result": result
    }

def final_judgment(state: AgentState) -> AgentState:
    """최종 판정: 3단계 결과 종합 및 신뢰도 점수 계산"""
    print("\n⚖️ [Final Judgment] 최종 판정 시작...")
    
    step1 = state.get("step1_result", {}) or {}
    step2 = state.get("step2_result", {}) or {}
    step3 = state.get("step3_result", {}) or {}
    
    # 각 단계별 점수 추출
    step1_score = step1.get("confidence", 0) if not step1.get("error") else 0
    step2_score = step2.get("confidence", 0) if not step2.get("error") else 0
    step3_score = step3.get("confidence", 0) if not step3.get("error") else 0
    
    # 핵심 지표 확인
    internal_heating_detected = step1.get("internal_heating_detected", False)
    swelling_detected = step2.get("swelling_detected", False) or step2.get("spongy_texture_detected", False)
    global_aging_detected = step3.get("global_aging_detected", False)
    conductor_fusion = step1.get("conductor_fusion", False)
    
    # 신뢰도 점수 계산 (가중치 적용)
    base_score = 0
    
    # 핵심 지표 가중치
    if internal_heating_detected: base_score += 35
    if conductor_fusion: base_score += 20
    if swelling_detected: base_score += 25
    if global_aging_detected: base_score += 20
    
    # 각 단계별 신뢰도 점수의 평균 반영 (10%)
    avg_confidence = (step1_score + step2_score + step3_score) / 3
    base_score += avg_confidence * 0.1
    
    # 핵심 3가지가 모두 확인되면 90% 이상 보장
    if internal_heating_detected and swelling_detected and global_aging_detected:
        base_score = max(base_score, 90)
    
    final_score = min(100, max(0, int(base_score)))
    
    # 증거 수집
    evidence = []
    if internal_heating_detected:
        evidence.append({"step": 1, "evidence": "내부 발열 확인", "details": step1.get("fusion_description", "")})
    if conductor_fusion:
        evidence.append({"step": 1, "evidence": "도체 융착 확인", "details": step1.get("fusion_description", "")})
    if swelling_detected:
        evidence.append({"step": 2, "evidence": "스펀지/부풀어 오름 현상 확인", "details": step2.get("texture_description", "")})
    if global_aging_detected:
        evidence.append({"step": 3, "evidence": "광역적 노후화 확인", "details": step3.get("discoloration_pattern", "")})
    
    # 분석 요약 생성
    summary_parts = [f"절연열화 판정 신뢰도: {final_score}%"]
    summary_parts.append(f"✓ 내부 발열 확인: {step1.get('carbonization_direction', 'unknown')}" if internal_heating_detected else "✗ 내부 발열 미확인")
    summary_parts.append("✓ 도체 융착 확인" if conductor_fusion else "✗ 도체 융착 미확인")
    summary_parts.append("✓ 스펀지/부풀어 오름 현상 확인" if swelling_detected else "✗ 스펀지/부풀어 오름 현상 미확인")
    summary_parts.append(f"✓ 광역적 노후화 확인: {step3.get('widespread_cracking', False)}" if global_aging_detected else "✗ 광역적 노후화 미확인")
    
    analysis_summary = "\n".join(summary_parts)
    print(f"✅ [Final Judgment] 완료: 신뢰도 {final_score}%")
    
    return {
        **state,
        "confidence_score": final_score,
        "analysis_summary": analysis_summary,
        "evidence": evidence
    }

In [13]:
def create_agent_graph():
    """절연열화 판별 에이전트 그래프 생성"""
    workflow = StateGraph(AgentState)
    
    # 노드 추가
    workflow.add_node("step1_carbonization", step1_carbonization_depth)
    workflow.add_node("step2_swelling", step2_swelling_analysis)
    workflow.add_node("step3_global_aging", step3_global_aging)
    workflow.add_node("final_judgment", final_judgment)
    
    # 엣지 연결 (순차 실행)
    workflow.set_entry_point("step1_carbonization")
    workflow.add_edge("step1_carbonization", "step2_swelling")
    workflow.add_edge("step2_swelling", "step3_global_aging")
    workflow.add_edge("step3_global_aging", "final_judgment")
    workflow.add_edge("final_judgment", END)
    
    return workflow.compile()

# 전역 그래프 객체
try:
    agent_app = create_agent_graph()
    print("✅ Langgraph StateGraph 구성 완료")
except Exception as e:
    print(f"⚠️ 그래프 구성 실패 (Langgraph 미설치 등): {e}")
    agent_app = None

def analyze_insulation_degradation(image_path: str) -> dict:
    """전체 절연열화 분석 실행 함수"""
    if agent_app is None:
        return {"error": "Agent 그래프가 초기화되지 않았습니다."}

    initial_state: AgentState = {
        "image_path": image_path,
        "image_part": None,
        "step1_result": None,
        "step2_result": None,
        "step3_result": None,
        "confidence_score": 0,
        "analysis_summary": "",
        "evidence": []
    }
    
    print(f"\n{'='*60}\n🔍 절연열화 분석 시작: {image_path}\n{'='*60}")
    
    try:
        final_state = agent_app.invoke(initial_state)
        return {
            "confidence_score": final_state["confidence_score"],
            "analysis_summary": final_state["analysis_summary"],
            "step1_result": final_state["step1_result"],
            "step2_result": final_state["step2_result"],
            "step3_result": final_state["step3_result"],
            "evidence": final_state["evidence"]
        }
    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        return {"error": str(e)}

✅ Langgraph StateGraph 구성 완료


In [14]:
def test_single_step(step_name: Literal["step1", "step2", "step3"], image_path: str, prev_state: Optional[AgentState] = None):
    """
    특정 단계만 독립적으로 테스트하기 위한 함수
    """
    print(f"\n🧪 [Test] {step_name} 독립 실행 테스트 중...")
    
    if prev_state:
        state = prev_state.copy()
    else:
        state: AgentState = {
            "image_path": image_path,
            "image_part": None, # 노드 내부에서 생성됨
            "step1_result": None,
            "step2_result": None,
            "step3_result": None,
            "confidence_score": 0,
            "analysis_summary": "",
            "evidence": []
        }
    
    try:
        if step_name == "step1":
            result_state = step1_carbonization_depth(state)
            print("결과:", json.dumps(result_state["step1_result"], indent=2, ensure_ascii=False))
        elif step_name == "step2":
            result_state = step2_swelling_analysis(state)
            print("결과:", json.dumps(result_state["step2_result"], indent=2, ensure_ascii=False))
        elif step_name == "step3":
            result_state = step3_global_aging(state)
            print("결과:", json.dumps(result_state["step3_result"], indent=2, ensure_ascii=False))
        return result_state
    except Exception as e:
        print(f"❌ 테스트 실패: {e}")
        return None

In [15]:
if __name__ == "__main__":
    # 테스트할 이미지 경로 설정 (노트북 기준 상대 경로)
    TEST_IMAGE_PATH = "../data/Cu2O_Breeding.jpg"
    
    # 이미지 파일 존재 여부 확인
    if not os.path.exists(TEST_IMAGE_PATH):
        print(f"⚠️ 경고: 테스트 이미지 '{TEST_IMAGE_PATH}'가 없습니다. 경로를 확인하세요.")
    else:
        # 전체 분석 실행
        result = analyze_insulation_degradation(TEST_IMAGE_PATH)
        
        # 결과 출력
        print("\n" + "="*60)
        print("📊 분석 결과")
        print("="*60)
        print(result.get("analysis_summary", ""))
        print(f"\n신뢰도 점수: {result.get('confidence_score', 0)}%")
        print("\n증거:")
        for ev in result.get("evidence", []):
            print(f"  - Step {ev.get('step')}: {ev.get('evidence')}")


🔍 절연열화 분석 시작: ../data/Cu2O_Breeding.jpg

🔍 [Step 1] 탄화의 심도 분석 시작...

🔍 [Step 1] 응답 파트 개수: 1

📊 [Step 1] 응답 객체 속성:
  - Candidate 속성: avg_logprobs, citation_metadata, content, finish_message, finish_reason, from_dict, function_calls, grounding_metadata, index, logprobs_result...
📋 [Step 1] Finish reason: 1

💭 [Step 1] 모델 응답 텍스트:
------------------------------------------------------------
📝 [Thinking 과정]:
## 분석 프로세스 상세 설명 (Chain of Thought)

### 1단계: 시각적 요소 추출
*   **전체 구조:** 두 가닥의 구리선이 확인됩니다. 하나는 여러 가닥이 꼬인 연선(stranded wire)이고, 다른 하나는 단일 가닥의 단선(solid wire)입니다.
*   **연선 관찰:**
    *   오른쪽 끝부분은 베이지색의 온전한 절연 피복이 관찰됩니다.
    *   중앙부로 오면서 피복이 소실되고, 검게 탄화된 부서지기 쉬운 형태의 물질(탄화된 절연체)이 나타납니다.
    *   이 탄화 물질은 도체(구리선)로부터 쉽게 벗겨지거나 이미 떨어져 나간 상태이며, 그 결과 구리 도체가 노출되어 있습니다.
    *   탄화가 가장 심한 부분은 절연체가 완전히 사라져 구리선이 그대로 드러나 있습니다.
*   **단선 관찰:**
    *   연선 위쪽에 위치한 단선은 중앙부의 절연 피복이 소실되고 구리 도체 표면이 검게 변색된 상태입니다.
*   **접속부 관찰:**
    *   왼쪽 끝에는 녹색의 반투명 물질(커넥터 또는 마감재로 추정)이 있으며, 그 주변으로 검은색의 탄화물이 관찰됩니다.

### 2단계: 특징 서술
